# 05 · Validation Plan — express, ELISA/SPR (binding), DSF (stability), controls

**Standard slot:** *validation plan.* **For Project 16 this means:** a wet-lab plan to test whether the
humanized variants **retain binding** (ELISA / SPR) and **stay stable** (DSF), plus an
**immunogenicity-risk summary**, with the two mandatory **controls** — the **parental** (non-human)
antibody and an **over-humanized decoy** (D4/D5).

This generates structured plan files and a costed-reagent stub; it runs with no GPU. The deliverable is
**variants + the trade-off analysis + the experiment** — a humanness score is a hypothesis until the
assays confirm retained binding and stability.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Why this experiment (not "we humanized it")

A humanized sequence is a **hypothesis**. A high humanness score does **not** prove low immunogenicity,
and grafting frequently **loses binding/stability** — that is exactly why the campaign produced *several*
variants (grafts, back-mutated grafts, resurfaced) plus controls. The experiment answers two questions
per variant: **does it still bind the antigen** (ELISA/SPR vs the parental) and **is it still stable**
(DSF Tm vs the parental)? The **over-humanized decoy** is the negative control that proves the assay can
detect over-humanization (it should lose binding and/or Tm).

## 1 · The binding + stability assay plan `[core]`

ELISA (fast yes/no binding), SPR/BLI (quantitative K_D), and DSF (thermal stability, Tm). The plan
generator records the stages, readouts, and controls so it is reproducible and gradable.

In [ ]:
import json, os

assay_plan = {
    "goal": "Test whether humanized variants RETAIN antigen binding and stability vs the parental.",
    "variants_under_test": "results/proj16_ranked.csv survivors (grafts, back-mutated grafts, resurfaced)",
    "expression": {
        "format": "Fab or full IgG (mammalian, e.g. ExpiCHO/HEK293) for binding/DSF; "
                  "scFv/Fab in E. coli periplasm acceptable for early ELISA triage",
        "note": "express each variant + BOTH controls under identical conditions",
    },
    "assays": [
        {"name": "ELISA", "measures": "qualitative antigen binding (retained vs lost)",
         "readout": "OD450 titration vs antigen-coated plate; EC50 estimate"},
        {"name": "SPR or BLI", "measures": "quantitative affinity (K_D, k_on, k_off)",
         "readout": "kinetics vs immobilized antigen; compare K_D to the parental"},
        {"name": "DSF (thermal shift)", "measures": "stability (melting temperature Tm)",
         "readout": "SYPRO-Orange Tm; compare Tm to the parental (humanization often lowers Tm)"},
    ],
    "decision": "KEEP variants that retain binding (K_D within an agreed fold of parental) AND keep Tm "
                "within an agreed margin; these balance humanness and developability.",
    "controls": {
        "positive_parental": "the PARENTAL non-human antibody — defines retained-binding (best K_D) and "
                             "the stability ceiling (Tm); every humanized variant is judged against it",
        "negative_over_humanized_decoy": "the OVER-HUMANIZED DECOY (humanized including the Vernier zone, "
                                         "no rescue) — expected to LOSE binding and/or Tm; proves the "
                                         "assay detects over-humanization",
        "negative_isotype": "an irrelevant isotype-matched antibody — no specific antigen binding",
    },
    "expectation": "Humanization commonly costs some affinity/stability; Vernier back-mutated and "
                   "resurfaced variants should recover more than the over-humanized decoy.",
}
os.makedirs("results", exist_ok=True)
with open("results/validation_plan.json", "w") as fh:
    json.dump(assay_plan, fh, indent=2)
print("wrote results/validation_plan.json")
for a in assay_plan["assays"]:
    print(f"  {a['name']:12s} -> {a['measures']}")
print("\ncontrols:")
for k, v in assay_plan["controls"].items():
    print(f"  {k}: {v}")

## 2 · Immunogenicity-risk summary `[core]`

The clinical motivation. Summarize each variant's residual immunogenicity risk: humanness band, residual
non-human framework content, any non-human residues re-introduced by back-mutations at exposed positions,
and (real run) predicted T-cell epitopes. State it as **risk**, paired with the *retained-binding* result
— a maximally-human variant that does not bind is useless. The honest framing: humanness reduces, but
does not eliminate, ADA risk; only a clinical immunogenicity assessment is definitive.

In [ ]:
import json
immuno = {
    "summary": "Per-variant residual-immunogenicity risk, paired with the retained-binding result.",
    "inputs": "results/immunogenicity_risk_summary.csv (humanness band + residual non-human FR content)",
    "axes": [
        "humanness band (OASis/Hu-mAb/T20 — use the REAL tools to report)",
        "residual non-human framework residues (count + exposed?)",
        "non-human residues re-introduced by Vernier back-mutations (exposed positions are higher risk)",
        "predicted T-cell epitopes (add a real predictor, e.g. NetMHCII-style, for a reportable claim)",
    ],
    "honesty": "Humanness scores correlate with, but do NOT prove, low ADA. Only a clinical "
               "immunogenicity assessment is definitive. Pair every humanness claim with the assay result.",
    "decision_rule": "Prefer the MOST human variant that still PASSES the binding (ELISA/SPR) and "
                     "stability (DSF) bars vs the parental. Do not chase humanness past loss of function.",
}
with open("results/immunogenicity_risk_plan.json", "w") as fh:
    json.dump(immuno, fh, indent=2)
print(json.dumps(immuno, indent=2))

## 3 · Controls (mandatory) — parental + over-humanized decoy `[core]`

Controls are non-negotiable, even in the plan. The **parental** (non-human) antibody is the
positive/retained-binding reference and the stability ceiling. The **over-humanized decoy** is the
negative control proving the assay detects over-humanization. An **isotype** control rules out
non-specific binding.

In [ ]:
controls = {
    "positive_parental": "PARENTAL non-human antibody — best binding + highest Tm; the benchmark every "
                         "humanized variant is compared against (retained binding = within agreed fold).",
    "negative_over_humanized_decoy": "OVER-HUMANIZED DECOY — humanized including the Vernier zone with no "
                                     "back-mutation rescue; expected to LOSE binding and/or Tm. If it "
                                     "still binds well, your assay (or your Vernier set) needs revisiting.",
    "negative_isotype": "irrelevant isotype-matched antibody — controls for non-specific binding.",
    "why_decoy": "Without an over-humanized decoy you cannot show your assay distinguishes 'human enough' "
                 "from 'too humanized to bind' — the central risk of this project.",
}
with open("results/controls.json", "w") as fh:
    json.dump(controls, fh, indent=2)
print(json.dumps(controls, indent=2))

## 4 · Costed reagent + timeline stub `[extension]`

A skeleton the student fills with real quotes/timelines. Numbers below are **EXAMPLE_DATA placeholders**
(not real quotes) — replace with vendor quotes in your D4 plan.

In [ ]:
import pandas as pd
plan_items = pd.DataFrame([
    dict(item="Gene synthesis (variants + 2 controls)", purpose="express each variant", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Mammalian expression (ExpiCHO/HEK)", purpose="Fab/IgG production", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Protein A/affinity purification", purpose="purify antibodies", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="Recombinant antigen (ELISA/SPR)", purpose="binding assays", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="ELISA reagents + plates", purpose="binding triage", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="SPR/BLI chip time", purpose="K_D vs parental", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="DSF (SYPRO-Orange, qPCR plate)", purpose="Tm stability", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
    dict(item="(Real run) humanness tools / T-cell epitope predictor", purpose="immunogenicity risk", est_cost_usd="EXAMPLE_DATA", weeks="EXAMPLE_DATA"),
])
plan_items.to_csv("results/experimental_plan.csv", index=False)
print("wrote results/experimental_plan.csv (EXAMPLE_DATA placeholders — fill with real quotes)")
plan_items

## Responsible research (state in the plan)

This is a **therapeutic antibody** project — **reducing immunogenicity** (ADA risk) of a non-human
therapeutic antibody so it is safer/more effective in patients. This is a defensive, low-dual-use aim.
In scope: humanizing therapeutic/diagnostic antibodies. Out of scope: enhancing pathogen
transmissibility/virulence, toxins, or any design intended to cause harm. Any real gene-synthesis order
must go through a biosecurity-screening provider (IGSC member); wet-lab work requires institutional
biosafety/ethics approval. Do not overstate a humanness score as a proven low-ADA outcome — only a
clinical immunogenicity assessment is definitive. See `MASTER_BLUEPRINT.md §7`.

## D4 / D5 checklist
- [ ] `results/validation_plan.json`: ELISA + SPR/BLI + DSF plan with stages, readouts, decision rule.
- [ ] `results/immunogenicity_risk_plan.json` + `immunogenicity_risk_summary.csv`: residual risk,
      paired with the retained-binding result, framed as risk (not a guarantee).
- [ ] `results/controls.json`: the **parental** + **over-humanized decoy** (+ isotype) controls, with
      why the decoy is essential.
- [ ] Costed reagent + timeline stub (EXAMPLE_DATA → real quotes).
- [ ] Responsible-research framing stated (immunogenicity reduction; low dual-use).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — this project follows the **antibody-family pattern** (Project 17's template): the
`design_type="antibody"` filter hand-off, a deterministic mock, and a controlled validation plan.